# Rosetta — Comparaison des tokenisations (étapes 4, 5, 5bis)

**Rôle de ce notebook** : construire les **6 configurations de tokenisation**
retenues pour le projet (`docs/plan-seq2seq.md`, étape 4 pour le vocabulaire,
étape 5 pour la numérisation, étape 5bis pour la caractérisation), et produire
le tableau des 5 stats clés ainsi que les 3 panneaux de comparaison BPE vs
Unigram demandés par l'étape 5bis (voir `docs/plan-seq2seq.md`, section
« Étape 5bis — Caractérisation par config de tokenisation »).

## Les 6 configurations comparées

| clé | type | détail |
|---|---|---|
| `full` | mots entiers | vocabulaire complet (couverture 100 %) |
| `words95` | mots entiers | coupé à 95 % de couverture des occurrences |
| `bpe4k` / `bpe8k` | SentencePiece BPE | 4 000 / 8 000 sous-mots |
| `unigram4k` / `unigram8k` | SentencePiece Unigram | 4 000 / 8 000 sous-mots |

Chaque vocabulaire est construit sur le **TRAIN seul** (anti-fuite, étape 4) via
`src.tokenization.vocab_builder.build_vocabs`. Le tokeniseur **entre dans
l'espace de recherche Optuna** (revient sur le choix initial de l'étape 8) :
ces 6 configs sont caractérisées ici, en amont, une fois pour toutes -- ce sont ces
statistiques qui permettent d'interpréter les essais Optuna qui choisiront parmi elles.

**Ce que ce notebook NE fait PAS** : il n'entraîne aucun modèle de traduction
(étapes 6-10) ; il caractérise uniquement l'effet du tokeniseur sur les
données, pour informer le choix de config utilisé plus tard par
`notebooks/Rosetta_Modelisation.ipynb` (`tokenizationConfig` y est désormais un hyperparamètre de l'espace de recherche Optuna, pas un paramètre fixé).


In [ ]:
# Paramètres
configNames = ["full", "words95", "bpe4k", "bpe8k", "unigram4k", "unigram8k"]  # doit correspondre à TOKENIZATION_CONFIGS
wordConfigs = ["full", "words95"]
subwordConfigs = ["bpe4k", "bpe8k", "unigram4k", "unigram8k"]

# Étape 5bis : rapporter aussi val/test (pas seulement le train) pour vérifier
# l'homogénéité du split -- sinon le point de contrôle "homogénéité" n'a rien à vérifier.
statsParts = ("train", "val", "test")

forceRebuild = False  # True pour ré-entraîner les 6x2 vocabulaires même si déjà en cache sur disque

# Chemins (relatifs à notebooks/, comme dans l'AED et dans Rosetta_Modelisation)
dataDir = "../data"  # corpus brut (repli OPUS), utilisé seulement si le cache de splits est absent
processedDir = "../data/processed"  # cache des splits (étapes 0-3) + modèles SentencePiece (data/ est gitignoré)
reportsDir = "../reports/tokenization"  # CSV + figures produits par ce notebook

# Ajustement de la pente de Zipf (même approche que le notebook AED, cellule "Courbe de Zipf") :
# hors tête (mots ultra-fréquents) et hors queue d'hapax.
zipfRankMin = 10
zipfRankMax = 4000

# Écart relatif maximum toléré entre train/val/test sur la longueur moyenne en
# tokens (point de contrôle d'homogénéité, étape 5bis).
homogeneiteTolerance = 0.05

figSize = (9, 5.5)


In [ ]:
# Imports et configuration globale
# Rendre `src/` importable quel que soit le dossier depuis lequel le kernel est
# lance (Jupyter demarre generalement dans notebooks/). Sans cela, les imports
# `src.*` ci-dessous echoueraient.
import sys
from pathlib import Path

projectRoot = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()),
    Path.cwd(),
)
if str(projectRoot) not in sys.path:
    sys.path.insert(0, str(projectRoot))

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.data.splits import load_splits
from src.tokenization.stats import build_stats_table, token_char_lengths, zipf_frequencies
from src.tokenization.vocab_builder import TOKENIZATION_CONFIGS, build_vocabs

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

Path(reportsDir).mkdir(parents=True, exist_ok=True)

assert list(TOKENIZATION_CONFIGS) == configNames, (
    "configNames (cellule Paramètres) doit correspondre aux clés de TOKENIZATION_CONFIGS "
    "(src/tokenization/vocab_builder.py) -- vérifier les deux en cas d'échec."
)

print(f"pandas: {pd.__version__} | numpy: {np.__version__}")
print(f"Configs à construire: {configNames}")


## 1. Chargement des splits (étapes 0→3)

Relit le cache `data/processed/{train,val,test}.parquet` s'il existe déjà
(anti-fuite déjà validée à l'étape 3 : split groupé par cible EN, stratifié par
longueur). Les 6 vocabulaires sont construits sur `splits["train"]` uniquement.


In [ ]:
# Chargement des splits (étapes 0→3, mis en cache dans data/processed/)
tStart = time.time()
splits = load_splits(data_dir=dataDir, processed_dir=processedDir)
print(
    f"Split -- train: {len(splits['train']):,} paires, val: {len(splits['val']):,} paires, "
    f"test: {len(splits['test']):,} paires ({time.time() - tStart:.1f}s)"
)
display(splits["train"].head(3))


## 2. Construction des 6 configurations de tokenisation (étape 4)

`build_vocabs` construit un vocabulaire FR et un vocabulaire EN par config, sur
le TRAIN seul. Les modèles SentencePiece sont mis en cache sur disque
(`data/processed/tokenizers/`) : une ré-exécution avec `forceRebuild=False` les
recharge au lieu de les ré-entraîner.


In [ ]:
# Construction des 6 configurations de tokenisation (étape 4), avec cache et progression
vocabs = {}
tStart = time.time()
for i, configName in enumerate(configNames, start=1):
    tConfig = time.time()
    print(f"[{i}/{len(configNames)}] config={configName} ...")
    frVocab, enVocab = build_vocabs(
        splits["train"], config=configName, processed_dir=processedDir, force_rebuild=forceRebuild
    )
    vocabs[configName] = (frVocab, enVocab)
    print(f"    -> vocab FR={len(frVocab)}, vocab EN={len(enVocab)} ({time.time() - tConfig:.1f}s)")

dureeConstruction = time.time() - tStart
print(f"\nLes {len(configNames)} configurations sont prêtes ({dureeConstruction:.1f}s au total).")


## 3. Tableau des 5 stats clés (étape 5bis)

Pour chaque config x langue x part (train/val/test) : longueur en tokens
(moyenne, médiane, p95, p99, max), ratio tokens/mots (fragmentation),
vocabulaire effectif, taux d'OOV et couverture. Sauvegardé dans
`reports/tokenization/stats_tokenisation.csv`.


In [ ]:
# Tableau des 5 stats clés (étape 5bis), pour les 6 configs x 2 langues x (train/val/test)
tStart = time.time()
statsTable = build_stats_table(splits, vocabs, parts=statsParts)
statsPath = Path(reportsDir) / "stats_tokenisation.csv"
statsTable.to_csv(statsPath, index=False)
print(f"Tableau des stats -> {statsPath} ({time.time() - tStart:.1f}s, {len(statsTable)} lignes)")
display(statsTable.round(3))


## 4. Comparaison Zipf BPE vs Unigram (panneau 1)

Courbes superposées en log-log (une par config sous-mots), avec pente ajustée
sur les rangs `[zipfRankMin, zipfRankMax]` (hors tête, hors queue d'hapax --
même approche que le notebook AED). Les tokens spéciaux (0-3) sont exclus.


In [ ]:
# Panneau 1 -- Zipf superposé BPE vs Unigram (log-log), pentes ajustées
def fitZipfExponent(freqs: np.ndarray, rankMin: int, rankMax: int) -> float:
    """Pente de la droite log-log, ajustée hors tête et hors queue d'hapax
    (même approche que le notebook AED, cellule "Courbe de Zipf")."""
    high = min(rankMax, len(freqs))
    if high <= rankMin:
        return float("nan")
    ranks = np.arange(rankMin, high + 1)
    slope, _ = np.polyfit(np.log10(ranks), np.log10(freqs[rankMin - 1 : high]), 1)
    return float(slope)


subwordStyles = {
    "bpe4k": {"color": "tab:blue", "linestyle": "-"},
    "bpe8k": {"color": "tab:blue", "linestyle": "--"},
    "unigram4k": {"color": "tab:orange", "linestyle": "-"},
    "unigram8k": {"color": "tab:orange", "linestyle": "--"},
}

fig, axes = plt.subplots(1, 2, figsize=(figSize[0] * 2, figSize[1]))
for ax, lang in zip(axes, ("fr", "en")):
    langIdx = 0 if lang == "fr" else 1
    for configName in subwordConfigs:
        vocab = vocabs[configName][langIdx]
        ranks, freqs = zipf_frequencies(vocab, splits["train"][lang])
        slope = fitZipfExponent(freqs, zipfRankMin, zipfRankMax)
        style = subwordStyles[configName]
        ax.loglog(ranks, freqs, linewidth=1.5, label=f"{configName} (pente {slope:.2f})", **style)
    ax.set_xlabel("rang du token (log)")
    ax.set_ylabel("fréquence (log)")
    ax.set_title(f"Zipf -- {lang}")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.25)

fig.suptitle("Panneau 1 -- Zipf superposé BPE vs Unigram (train)", fontweight="bold")
plt.tight_layout()
figPath = Path(reportsDir) / "fig_zipf_bpe_vs_unigram.png"
fig.savefig(figPath, bbox_inches="tight")
plt.show()
print(f"Figure -> {figPath}")


## 5. Distribution des longueurs de tokens en caractères (panneau 2)

Le plan est explicite : ce panneau est **souvent plus informatif que le Zipf
lui-même**, car c'est là que la différence de mécanisme se voit -- Unigram tend
vers des tokens plus longs / plus morphémiques, BPE vers des fusions plus
opportunistes. Le marqueur de début de mot SentencePiece (`▁`) est retiré avant
mesure ; les tokens spéciaux sont exclus.


In [ ]:
# Panneau 2 -- distribution des longueurs de tokens en caractères, BPE vs Unigram
fig, axes = plt.subplots(1, 2, figsize=(figSize[0] * 2, figSize[1]))
for ax, lang in zip(axes, ("fr", "en")):
    langIdx = 0 if lang == "fr" else 1
    maxLength = 1
    for configName in subwordConfigs:
        vocab = vocabs[configName][langIdx]
        lengths = token_char_lengths(vocab, splits["train"][lang])
        maxLength = max(maxLength, int(lengths.max()))
        style = subwordStyles[configName]
        ax.hist(
            lengths,
            bins=np.arange(0.5, maxLength + 1.5, 1),
            density=True,
            histtype="step",
            linewidth=1.8,
            label=configName,
            color=style["color"],
            linestyle=style["linestyle"],
        )
    ax.set_xlabel("longueur du token (caractères)")
    ax.set_ylabel("densité")
    ax.set_title(f"Longueur des tokens -- {lang}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

fig.suptitle("Panneau 2 -- distribution des longueurs de tokens (BPE vs Unigram, train)", fontweight="bold")
plt.tight_layout()
figPath = Path(reportsDir) / "fig_longueur_tokens_caracteres.png"
fig.savefig(figPath, bbox_inches="tight")
plt.show()
print(f"Figure -> {figPath}")


## 6. Fragmentation (tokens/mot) par méthode (panneau 3)

Un ratio `tokens/mot` de 1.0 est attendu pour les configs mots entiers (`full`,
`words95`), et `> 1.0` pour les 4 configs sous-mots -- c'est la variable qui
gouverne l'allongement des séquences (et donc le risque de décrochage RNN, cf.
étape 6).


In [ ]:
# Panneau 3 -- fragmentation (tokens/mot) par méthode
fragTrain = statsTable[statsTable["part"] == "train"]
methodOf = {
    "full": "mots entiers",
    "words95": "mots entiers",
    "bpe4k": "BPE",
    "bpe8k": "BPE",
    "unigram4k": "Unigram",
    "unigram8k": "Unigram",
}
methodColors = {"mots entiers": "tab:gray", "BPE": "tab:blue", "Unigram": "tab:orange"}

fig, axes = plt.subplots(1, 2, figsize=(figSize[0] * 2, figSize[1]))
for ax, lang in zip(axes, ("fr", "en")):
    sub = fragTrain[fragTrain["lang"] == lang].set_index("config").loc[configNames]
    colors = [methodColors[methodOf[c]] for c in configNames]
    ax.bar(configNames, sub["ratio_tokens_mots"], color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(1.0, color="black", linestyle=":", linewidth=1)
    ax.set_xticks(range(len(configNames)))
    ax.set_xticklabels(configNames, rotation=20)
    ax.set_ylabel("ratio tokens / mots")
    ax.set_title(f"Fragmentation -- {lang}")
    ax.grid(True, axis="y", alpha=0.25)

legendHandles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in methodColors.values()]
fig.legend(legendHandles, methodColors.keys(), loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("Panneau 3 -- fragmentation (tokens/mot) par méthode (train)", fontweight="bold", y=1.12)
plt.tight_layout()
figPath = Path(reportsDir) / "fig_fragmentation.png"
fig.savefig(figPath, bbox_inches="tight")
plt.show()
print(f"Figure -> {figPath}")


## 7. Points de contrôle (étape 5bis)

Vérifie, avec des `assert` : le ratio tokens/mots (1.0 pour mots entiers, > 1
pour les sous-mots) et l'homogénéité train/val/test sur la longueur moyenne en
tokens (pas de décalage de distribution introduit par le split, étape 3).


In [ ]:
# Points de contrôle -- étape 5bis du plan
print("=" * 74)
print("Points de contrôle -- étape 5bis")
print("=" * 74)

trainStats = statsTable[statsTable["part"] == "train"]

for configName in wordConfigs:
    ratios = trainStats.loc[trainStats["config"] == configName, "ratio_tokens_mots"]
    assert np.allclose(ratios, 1.0), f"{configName}: ratio tokens/mots attendu = 1.0, obtenu {ratios.tolist()}"
print("[OK] ratio tokens/mots = 1.0 pour les configs mots entiers (full, words95)")

for configName in subwordConfigs:
    ratios = trainStats.loc[trainStats["config"] == configName, "ratio_tokens_mots"]
    assert (ratios > 1.0).all(), f"{configName}: ratio tokens/mots attendu > 1.0, obtenu {ratios.tolist()}"
print("[OK] ratio tokens/mots > 1.0 pour les 4 configs sous-mots (BPE/Unigram)")

# Vocabulaire FR nettement > vocabulaire EN (asymétrie morphologique) --
# ce point de contrôle ne vaut QUE pour les configs mots entiers (vocabulaires séparés).
for configName in wordConfigs:
    frLen, enLen = len(vocabs[configName][0]), len(vocabs[configName][1])
    assert frLen > enLen, (
        f"{configName}: vocabulaire FR ({frLen}) attendu nettement > vocabulaire EN ({enLen})"
    )
print("[OK] vocabulaire FR nettement > vocabulaire EN pour les configs mots entiers (full, words95)")

# Pendant du point de contrôle ci-dessus pour les configs sous-mots : vocabulaire
# CONJOINT (même objet FR/EN) et taille = vocab_size visé.
for configName in subwordConfigs:
    frVocabConfig, enVocabConfig = vocabs[configName]
    assert frVocabConfig is enVocabConfig, f"{configName}: vocabulaire conjoint attendu (fr_vocab is en_vocab)"
    vocabSizeVise = TOKENIZATION_CONFIGS[configName]["vocab_size"]
    assert len(frVocabConfig) == len(enVocabConfig) == vocabSizeVise, (
        f"{configName}: taille de vocabulaire conjoint attendue = {vocabSizeVise}, "
        f"obtenu FR={len(frVocabConfig)}, EN={len(enVocabConfig)}"
    )
print("[OK] vocabulaire conjoint (même objet, taille = vocab_size visé) pour les 4 configs sous-mots")

if set(("train", "val", "test")).issubset(set(statsParts)):
    ecarts = []
    for configName in configNames:
        for lang in ("fr", "en"):
            sub = statsTable[(statsTable["config"] == configName) & (statsTable["lang"] == lang)]
            valeurs = sub.set_index("part")["longueur_moyenne"]
            ecart = (valeurs.max() - valeurs.min()) / valeurs.mean()
            ecarts.append({"config": configName, "lang": lang, "ecart_relatif": ecart})
    ecartsDf = pd.DataFrame(ecarts)
    ecartMax = float(ecartsDf["ecart_relatif"].max())
    assert ecartMax <= homogeneiteTolerance, (
        f"Écart train/val/test trop élevé (longueur moyenne en tokens) : "
        f"{ecartMax:.2%} > {homogeneiteTolerance:.0%}\n{ecartsDf.sort_values('ecart_relatif', ascending=False).head()}"
    )
    print(
        f"[OK] homogénéité train/val/test (longueur moyenne en tokens) : "
        f"écart relatif max {ecartMax:.2%} <= {homogeneiteTolerance:.0%}"
    )
else:
    print("[SKIP] homogénéité train/val/test non vérifiée (statsParts ne couvre pas les 3 parts)")

print("\nTous les points de contrôle de l'étape 5bis sont validés.")


## 8. Synthèse

À compléter après lecture des chiffres et figures ci-dessus : ce que le
tableau des 5 stats clés et les 3 panneaux disent du compromis de chaque
config (vocabulaire vs longueur de séquence vs OOV), et si la forme de Zipf ou
la longueur des tokens est le facteur le plus discriminant entre BPE et
Unigram sur ce corpus. Conclusion honnête attendue par le plan : si les
distributions Zipf s'avèrent proches, le dire plutôt que de forcer un résultat.
